In [1]:
import pandas as pd

# 1. 파일 불러오기 (경로는 필요에 따라 수정)
file_path = "./archive/2020 전국 기후.csv"

# 2. CSV 파일 읽기 (cp949 인코딩)
df = pd.read_csv(file_path, encoding='cp949')
df.columns = df.columns.str.strip()  # 혹시 모를 공백 제거

# 3. 일시 컬럼을 datetime 형식으로 변환
df['일시'] = pd.to_datetime(df['일시'], format='%Y.%m.%d %H:%M', errors='coerce')
df = df.dropna(subset=['일시'])

# 4. 평균 낼 수 있는 수치형 컬럼 지정
numeric_cols = ['기온(°C)', '습도(%)', '지면온도(°C)']
numeric_cols = [col for col in numeric_cols if col in df.columns]

# 5. 일시별로 평균 계산 (전국 평균)
df_avg = df.groupby('일시')[numeric_cols].mean().reset_index()

# 6. 결과 저장 (선택사항)
df_avg.to_csv("./archive/2020_climate_avg.csv", index=False, encoding='utf-8-sig')  # 로컬 저장

# 7. 출력 (선택사항)
print(df_avg.head())


                   일시    기온(°C)      습도(%)  지면온도(°C)
0 2020-01-01 01:00:00 -5.229474  53.694737 -3.950000
1 2020-01-01 02:00:00 -5.210526  55.400000 -3.927368
2 2020-01-01 03:00:00 -5.061053  57.105263 -3.767368
3 2020-01-01 04:00:00 -4.854737  59.410526 -3.546316
4 2020-01-01 05:00:00 -4.336842  61.052632 -3.205263


In [2]:
import pandas as pd
import os

# 처리할 연도 목록
years = ['2020', '2021', '2022', '2023']

# 평균을 낼 컬럼
numeric_cols = ['기온(°C)', '습도(%)', '지면온도(°C)']

for year in years:
    file_name = f"./archive/{year} 전국 기후.csv"
    if not os.path.exists(file_name):
        print(f"{file_name} 파일이 없습니다.")
        continue

    # 1. CSV 파일 읽기
    df = pd.read_csv(file_name, encoding='cp949')
    df.columns = df.columns.str.strip()

    # 2. 일시 컬럼 datetime 변환
    df['일시'] = pd.to_datetime(df['일시'], format='%Y.%m.%d %H:%M', errors='coerce')
    df = df.dropna(subset=['일시'])

    # 3. 존재하는 컬럼만 포함
    valid_cols = [col for col in numeric_cols if col in df.columns]

    # 4. 일시 기준 평균 계산
    df_avg = df.groupby('일시')[valid_cols].mean().reset_index()

    # 🔹 5. 소수점 첫째 자리로 반올림
    df_avg[valid_cols] = df_avg[valid_cols].round(1)

    # 6. 저장
    output_name = f"./archive/{year}_climate_avg.csv"
    df_avg.to_csv(output_name, index=False, encoding='utf-8-sig')
    print(f"{output_name} 저장 완료")


./archive/2020_climate_avg.csv 저장 완료
./archive/2021_climate_avg.csv 저장 완료
./archive/2022_climate_avg.csv 저장 완료
./archive/2023_climate_avg.csv 저장 완료


In [3]:
import pandas as pd

# 병합할 연도
years = ['2020', '2021', '2022', '2023']

# 연도별 파일을 읽어서 리스트에 저장
df_list = []
for year in years:
    file_name = f"./archive/{year}_climate_avg.csv"
    try:
        df = pd.read_csv(file_name, encoding='utf-8-sig')
        df['일시'] = pd.to_datetime(df['일시'])
        df_list.append(df)
    except Exception as e:
        print(f"{file_name} 불러오기 실패: {e}")

# 하나로 병합
merged_df = pd.concat(df_list, ignore_index=True)

# 시간 순 정렬
merged_df = merged_df.sort_values(by='일시').reset_index(drop=True)

# 저장
merged_df.to_csv("./archive/climate_2020_2023_merged.csv", index=False, encoding='utf-8-sig')
print("✅ climate_2020_2023_merged.csv 저장 완료")


✅ climate_2020_2023_merged.csv 저장 완료
